<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week2/DLP_Week2_GA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# !pip install -U "datasets==3.6.0"

In [55]:
from datasets import load_dataset, get_dataset_split_names, concatenate_datasets
from pprint import pprint

In [7]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

In [27]:
from transformers import AutoTokenizer

In [4]:
ds = load_dataset('bookcorpus/bookcorpus', split='all')
pprint(ds)

README.md:   0%|          | 0.00/18.5k [00:00<?, ?B/s]

bookcorpus.py:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

The repository for bookcorpus/bookcorpus contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bookcorpus/bookcorpus.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 74004228
})


In [5]:
ds = ds.select(range(0, len(ds), 7))
pprint(len(ds))

10572033


In [6]:
text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

Download the BookCorpus dataset. Take every 7-th sample (the indices are multiple of `7:[0,7,14,21,...])` from the entire dataset. This will result in a dataset with 10 million samples (exactly, 10,572,033). Use these samples to build a tokenizer with the BPE tokenization algorithm by varying the vocabulary size.

- Normalizer: LowerCase
- PreTokenizer: WhiteSpace
- Model: BPE
- Special tokens: [GO],[UNK],[PAD],[EOS]
- PostProcessing: None

Tokenize the input text: “SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.” using the following configurations

## Question 1: Keep the vocabulary size at 5000 and tokenize the input text using the learned vocabulary. Choose the number of tokens returned by the tokenizer.

In [8]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.normalizer = normalizers.Lowercase()
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(
    vocab_size=5000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
    continuing_subword_prefix="##"
)

In [9]:
def batch_iterator(batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i + batch_size]["text"]

tokenizer.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer
)

In [19]:
output = tokenizer.encode(text)
print(output)

Encoding(num_tokens=32, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [11]:
print("Tokens:", output.tokens)
print("Token IDs:", output.ids)
print("Token count:", len(output.ids))

Tokens: ['seb', '##i', 'study', 'find', '##s', '9', '##3', '%', 'of', 'ind', '##ivid', '##ual', 'f', '&', 'o', 'tr', '##ad', '##ers', 'made', 'loss', '##es', 'between', 'f', '##y', '##2', '##2', 'and', 'f', '##y', '##2', '##4', '.']
Token IDs: [3843, 87, 2739, 659, 83, 33, 103, 13, 152, 1129, 4472, 1136, 52, 14, 61, 393, 189, 306, 520, 3825, 151, 828, 52, 93, 102, 102, 142, 52, 93, 102, 108, 22]
Token count: 32


## Question 2: Increase the vocabulary size to 10K, 15K and 32K. For each case, tokenize the same input with the newly learned vocabulary. Choose all the correct statements

In [20]:
def train_tokenizer(vocab_size):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.normalizer = normalizers.Lowercase()
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
        continuing_subword_prefix="##"
    )

    tokenizer.train_from_iterator(
        batch_iterator(batch_size=10000),
        trainer=trainer
    )

    return tokenizer

for vocab_size in [5000, 10000, 15000, 32000]:
    tok = train_tokenizer(vocab_size)
    result = tok.encode(text)
    print(f"{vocab_size}: {len(result.ids)} tokens")

5000: 32 tokens
10000: 28 tokens
15000: 28 tokens
32000: 25 tokens


### Wrong Approach

In [14]:
trainer10k = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
    continuing_subword_prefix="##"
)
def batch_iterator(batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i + batch_size]["text"]

tokenizer.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer10k
)
output10k = tokenizer.encode(text)
print(f'10k Vocab Size: {output}')

10k Vocab Size: Encoding(num_tokens=32, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [17]:
trainer15k = trainers.BpeTrainer(
    vocab_size=15000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
    continuing_subword_prefix="##"
)
def batch_iterator(batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i + batch_size]["text"]

tokenizer.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer15k
)
output15k = tokenizer.encode(text)
print(f'15k Vocab Size: {output15k}')

15k Vocab Size: Encoding(num_tokens=32, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [18]:
trainer32k = trainers.BpeTrainer(
    vocab_size=32000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
    continuing_subword_prefix="##"
)
def batch_iterator(batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i + batch_size]["text"]

tokenizer.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer32k
)
output32k = tokenizer.encode(text)
print(f'32k Vocab Size: {output32k}')

15k Vocab Size: Encoding(num_tokens=32, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


Why this approach doesn't work?

In original code, a tokenizer is created once:

```
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
```
Then train it with a vocabulary size of 5,000:

```
tokenizer.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer
)
```
Next, you create a trainer with a vocabulary size of 10,000 and use it on the same tokenizer:

```
trainer10k = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
    continuing_subword_prefix="##"
)

tokenizer.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer10k
)
```

The problem is that you're attempting to train a tokenizer model that has already been trained.
A BPE tokenizer learns a vocabulary and merge rules from the training corpus. For example, it might learn these tokens:
- Vocabulary size 5K: study, find, individual
- Vocabulary size 10K: potentially study, finds, individual, traders

The larger vocabulary can contain more complete words or longer subword units, allowing text to be represented with fewer tokens.
The key point: Each experiment should start with a fresh, untrained BPE model so that the vocabulary size is the intended configuration for that experiment.

## Question 3: Download the pre-trained tokenizer file “hopper.json” used in the lecture. The tokenizer was trained on all 70 million samples in the BookCorpus dataset. Tokenize the same input text using this “hopper” tokenizer. How many tokens are there?

[After finding the answer, take a moment to compare the hopper tokenizer with the previous one]

In [23]:
trained = Tokenizer(models.BPE()).from_file('/content/hopper.json')
tokens = trained.encode(text).tokens
print(tokens)
print("Number of tokens: ", len(tokens) )

['seb', '##i', 'study', 'finds', '9', '##3', '%', 'of', 'individual', 'f', '&', 'o', 'traders', 'made', 'losses', 'between', 'f', '##y', '##22', 'and', 'f', '##y', '##2', '##4', '.']
Number of tokens:  25


## Question 4: Suppose we know that the acronym `“FY”` will likely appear very frequently in most of the input text (assume the text comes from the financial domain). Therefore, we hope that adding it manually to the vocabulary might help. Add the token “FY” to the vocabulary and tokenize (use the Hopper tokenizer) the input text. Enter the number of tokens produced.

In [25]:
trained.add_tokens(['FY'])
len(trained.encode(text))

22

## Question 5: Suppose that the context length of the model is 128. Assume that a mini-batch of size 8 samples is passed to a tokenizer that corresponds to a model from hub. After tokenization, the maximum length of sample in the batch is 64. The statement that zero is appended to the “input ids” of the remaining samples to make the length 64 is
 a) True

 b) False

 c) Insufficient info

The statement that zero is appended to the “input ids” of the remaining samples to make the length 64 is actually False.
Padding tokens are typically appended to the “input ids” of shorter sequences in a batch to make their lengths equal to the maximum sequence length in the batch. However, the specific padding token and its ID depend on the tokenizer being used. For instance, some models use a padding token like [PAD] with a specific token ID (often not zero), while others might use a different token or value. Without knowing the exact tokenizer and its padding strategy, we cannot assume that zero is the padding token.
Thus, the correct answer should be **Insufficient information to conclude**.

## Question 6: Load the “bert-base-uncased” and "gpt2” tokenizers (use AutoTokenizer function from transformers). Which of the following special tokens are used in these tokenizers? (MSQ)

- [ ] [GO]
- [ ] [CLS]
- [ ] [BOS]
- [ ] [SEP]
- [ ] [<|endoftext|>]

In [35]:
special_tokens_options = ['[GO]', '[CLS]', '[BOS]', '[SEP]', '<|endoftext|>']
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

In [33]:
bert_tokenizer.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [43]:
gpt2_tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}

In [48]:
bert_specials = set(bert_tokenizer.all_special_tokens)
gpt2_specials = set(gpt2_tokenizer.all_special_tokens)

for token in special_tokens_options:
  if token in bert_specials or token in gpt2_specials:
    print(token)

[CLS]
[SEP]
<|endoftext|>


## Question 7: By now, we have four tokenizers.

## 1. Custom tokenizer (vocab size 32K, trained on 10 million samples)
## 2. bert-base-uncased
## 3. gpt2
## 4. hopper

## Use these four tokenizers to count the number of tokens for the entire “imdb” dataset (drop the “unsupervised” part of the dataset). Enter the tokenizers in order such that the size of the dataset (measured in tokens) as returned by the tokenizers is in ascending order. For example, if the first tokenizer yields the smallest number of tokens and the fourth tokenizer yields the largest, you would enter 1234 (without any spaces).”

In [49]:
imdb = load_dataset("stanfordnlp/imdb")
pprint(imdb)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [50]:
imdb.pop('unsupervised')
pprint(imdb)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


In [58]:
splits = get_dataset_split_names('stanfordnlp/imdb')
splits.remove('unsupervised')
imdb_ds_concat = concatenate_datasets([imdb[split] for split in splits])
pprint(imdb_ds_concat)

Dataset({
    features: ['text', 'label'],
    num_rows: 50000
})


In [62]:
tokenizers32k = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizers32k.normalizer = normalizers.Lowercase()
tokenizers32k.pre_tokenizer = pre_tokenizers.Whitespace()
trainer32k = trainers.BpeTrainer(
    vocab_size=32000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"],
    continuing_subword_prefix="##"
)
def batch_iterator(batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i + batch_size]["text"]

tokenizers32k.train_from_iterator(
    batch_iterator(batch_size=10000),
    trainer=trainer32k
)
output32k = tokenizer.encode(text)
print(f'32k Vocab Size: {output32k}')

32k Vocab Size: Encoding(num_tokens=32, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [64]:
num_token = lambda tokenizer, input_text: len(tokenizer.encode(input_text))

In [65]:
imdn_tokens, c = {}, 1
for tokenizer in [tokenizers32k, bert_tokenizer, gpt2_tokenizer, trained]:
  imdn_tokens[c] = sum([num_token(tokenizer, text) for text in imdb_ds_concat['text']])
  c += 1

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


In [66]:
''.join(map(str, sorted(imdn_tokens.keys(), key = lambda x: imdn_tokens[x])))

'4312'

In [67]:

from tqdm.auto import tqdm

# Tokenizer mapping: index -> tokenizer
tokenizers = {
    1: tokenizers32k,
    2: bert_tokenizer,
    3: gpt2_tokenizer,
    4: trained
}

# Count tokens for a single text using either tokenizer library
def num_token(tokenizer, text):
    encoded = tokenizer.encode(text)
    return len(encoded.ids) if hasattr(encoded, "ids") else len(encoded)

# imdb_ds_concat should contain only train + test, excluding unsupervised
total_tokens = {}

for idx, tokenizer in tokenizers.items():
    total = 0

    for text in tqdm(imdb_ds_concat["text"], desc=f"Tokenizer {idx}"):
        total += num_token(tokenizer, text)

    total_tokens[idx] = total

# Display totals
print("\nTotal tokens:")
for idx, total in total_tokens.items():
    print(f"{idx}: {total:,}")

# Sort tokenizer indices by total token count, ascending
answer = "".join(
    str(idx)
    for idx, total in sorted(total_tokens.items(), key=lambda item: item[1])
)

print("\nAnswer:", answer)


Tokenizer 1:   0%|          | 0/50000 [00:00<?, ?it/s]

Tokenizer 2:   0%|          | 0/50000 [00:00<?, ?it/s]

Tokenizer 3:   0%|          | 0/50000 [00:00<?, ?it/s]

Tokenizer 4:   0%|          | 0/50000 [00:00<?, ?it/s]


Total tokens:
1: 15,352,840
2: 15,516,058
3: 14,812,432
4: 13,530,397

Answer: 4312


## Question 8: The statement that the special tokens and their respective token ids are model-specific (model here refers to a language model) is

- [ ] True
- [ ] False

Answer: True.

Special tokens and their token IDs are model-specific because different language models can use different token vocabularies and tokenization schemes.

For example:

| Model                      | Special token       | Token ID (typical) |
| -------------------------- | ------------------- | ------------------ |
| BERT (`bert-base-uncased`) | `[CLS]`             | 101                |
| BERT (`bert-base-uncased`) | `[SEP]`             | 102                |
| GPT-2 (`gpt2`)             | `<\\|endoftext\\|>` | 50256              |

BERT uses `[CLS]` and `[SEP]` for tasks such as classification and separating sequences, whereas GPT-2 uses `<|endoftext|>` as its end-of-text token.

Key takeaway: The same token string can have different IDs in different models, and a special token in one model might not be a special token in another.